# Intermediate 03 — Agent Identity and Delegated Authority

A Northwind support agent must read one case attachment through two service hops. You will keep the represented human, managed agent, and executing workloads distinct; issue a narrow root grant; exchange it for a narrower child grant; and enforce current authority at the resource boundary.

## Learning objectives and safety contract

By the end, you can explain subject, agent, actor, delegate, audience, and sender binding; prove that delegated authority only narrows; reject stale lifecycle and entitlement state; and preserve dependency failures as errors. All identities, grants, keys, resources, and content are synthetic. The HMAC grant and proof formats are teaching analogues—not production OAuth, DPoP, or workload-identity implementations.

In [ ]:
from dataclasses import replace
from datetime import timedelta
from pathlib import Path
import sys

COURSE = Path.cwd()
if not (COURSE / 'lab.py').exists():
    COURSE = Path('curriculum/roadmap/intermediate/03-agent-identity-and-delegated-authority').resolve()
assert (COURSE / 'lab.py').exists(), 'Run from the repository root or the course directory.'
sys.path.insert(0, str(COURSE))
import lab

print('Loaded course:', COURSE.name)
print('Synthetic clock:', lab.NOW.isoformat())

## 1. Follow authority across trust boundaries

![Agent identity and delegated authority architecture](architecture.svg)

The human is the **subject**. The managed agent is a governed application identity with sponsor and lifecycle. Each service workload is an authenticated **actor** and **delegate**. The **audience** names exactly one accepting resource server. A sender-bound proof demonstrates possession of the workload key for one request; it does not replace authorization.

In [ ]:
scenario = lab.build_scenario()
identity_contract = {
    'subject': scenario.identities.sessions['session:alice'].principal_id,
    'agent': scenario.identities.agents['agent:support-7'].agent_id,
    'initial_actor': scenario.identities.workloads['attest:support'].workload_id,
    'next_actor': scenario.identities.workloads['attest:case'].workload_id,
    'allowed_root_audiences': sorted(scenario.identities.agents['agent:support-7'].allowed_audiences),
    'current_entitlement_version': scenario.entitlements.snapshots['alice'].version,
}
identity_contract

## 2. Establish the claims-only failure mode

The deliberately unsafe baseline trusts model-controlled tenant and scope strings. It has no authenticated subject, workload, agent lifecycle, audience, resource entitlement, sender key, or current policy check.

In [ ]:
south_resource = scenario.resources['attachment:south:9']
baseline_accepts = lab.unsafe_claims_baseline(
    claimed_tenant='south', claimed_scopes=frozenset({'read'}), resource=south_resource
)
assert baseline_accepts
print('Unsafe baseline accepted a forged cross-tenant claim:', baseline_accepts)

## 3. Run the safe two-hop path

The application owns identity context and request IDs. The agent supplies only the attachment identifier. The case service verifies the root grant and fresh proof, authenticates as itself, and exchanges the root for a child grant addressed only to the attachment store.

In [ ]:
scenario = lab.build_scenario()
response = scenario.application.read_attachment(
    session_id='session:alice', agent_attestation_id='attest:support',
    attachment_id='attachment:north:7', request_id='notebook:valid', now=lab.NOW,
)
assert response.status is lab.DecisionStatus.ALLOW
assert response.content == 'Synthetic Northwind case attachment'
assert scenario.application.ambient_fallback_count == 0
[(receipt.operation, receipt.status, receipt.audience, receipt.actor_chain) for receipt in response.receipts]

In [ ]:
root, child = (scenario.grants.grants[grant_id] for grant_id in response.grant_ids)
assert child.root_grant_id == root.root_grant_id
assert child.parent_grant_id == root.grant_id
assert child.audience == 'attachment-store'
assert child.actor_chain == ('support-orchestrator', 'case-service')
{
    'root': (root.audience, sorted(root.operations), sorted(root.resources), root.confirmation_thumbprint),
    'child': (child.audience, sorted(child.operations), sorted(child.resources), child.confirmation_thumbprint),
}

## 4. Prove monotonic attenuation

A child may change audience and actor, but it must preserve subject, tenant, managed-agent identity, and purpose while narrowing operations, resources, expiry, and remaining delegation depth.

In [ ]:
scenario = lab.build_scenario()
parent = scenario.grants.issue(
    'session:alice', 'attest:support',
    lab.root_request(
        'notebook:parent', operations=frozenset({'read', 'comment'}),
        resources=frozenset({'attachment:north:7', 'attachment:north:8'}),
        ttl=timedelta(minutes=5),
    ), now=lab.NOW,
)
child = scenario.grants.exchange(
    parent.grant.grant_id, 'attest:case',
    lab.child_request('notebook:child', ttl=timedelta(hours=1)), now=lab.NOW,
)
assert child.status is lab.DecisionStatus.ALLOW
assert child.grant.operations < parent.grant.operations
assert child.grant.resources < parent.grant.resources
assert child.grant.expires_at <= parent.grant.expires_at
child.grant

## 5. Reject audience confusion and a stolen bearer

An attachment-store grant cannot be presented to the case service. Possessing the serialized grant is also insufficient: the caller must be the named delegate and prove possession of its bound key.

In [ ]:
nonce = scenario.case_boundary.challenge('notebook:audience')
proof = scenario.proofs.issue(
    'attest:case', child.grant, proof_id='proof:notebook:audience', method='GET',
    uri='https://case-service.internal/resources/attachment:north:7', nonce=nonce, now=lab.NOW,
)
confused = scenario.case_boundary.authorize(
    'attest:case', child.grant.grant_id, proof, operation='read',
    resource_id='attachment:north:7', request_id='notebook:audience', now=lab.NOW,
)
assert (confused.status, confused.reason) == (lab.DecisionStatus.DENY, 'audience')
confused.status.value, confused.reason

In [ ]:
nonce = scenario.attachment_store.challenge('notebook:stolen')
attacker_proof = scenario.proofs.issue(
    'attest:stolen-host', child.grant, proof_id='proof:notebook:stolen', method='GET',
    uri='https://attachment-store.internal/resources/attachment:north:7', nonce=nonce, now=lab.NOW,
)
stolen = scenario.attachment_store.authorize(
    'attest:stolen-host', child.grant.grant_id, attacker_proof, operation='read',
    resource_id='attachment:north:7', request_id='notebook:stolen', now=lab.NOW,
)
assert (stolen.status, stolen.reason) == (lab.DecisionStatus.DENY, 'sender')
stolen.status.value, stolen.reason

## 6. Bind and consume one request proof

The teaching proof is bound to grant digest, sender key, HTTP method, exact URI, server nonce, and time. A valid proof is consumed once—even if a later resource-policy check denies the request.

In [ ]:
nonce = scenario.attachment_store.challenge('notebook:proof')
one_use = scenario.proofs.issue(
    'attest:case', child.grant, proof_id='proof:notebook:once', method='GET',
    uri='https://attachment-store.internal/resources/attachment:north:7', nonce=nonce, now=lab.NOW,
)
first = scenario.attachment_store.authorize(
    'attest:case', child.grant.grant_id, one_use, operation='read',
    resource_id='attachment:north:7', request_id='notebook:proof:first', now=lab.NOW,
)
replay = scenario.attachment_store.authorize(
    'attest:case', child.grant.grant_id, one_use, operation='read',
    resource_id='attachment:north:7', request_id='notebook:proof:replay', now=lab.NOW,
)
assert first.status is lab.DecisionStatus.ALLOW
assert (replay.status, replay.reason) == (lab.DecisionStatus.DENY, 'proof-replay')
[(first.status.value, first.reason), (replay.status.value, replay.reason)]

## 7. Re-evaluate current lifecycle and entitlements

A grant records the decision basis, but it is not a permanent authorization. Resource servers consult current agent lifecycle, current entitlement version, active lineage state, and current policy before releasing data.

In [ ]:
scenario = lab.build_scenario()
issued = scenario.grants.issue('session:alice', 'attest:support', lab.root_request('notebook:stale'), now=lab.NOW)
current = scenario.entitlements.snapshots['alice']
scenario.entitlements.snapshots['alice'] = replace(current, version=current.version + 1)
entitlement, reason = scenario.grants.validate_current(issued.grant, now=lab.NOW)
assert entitlement is None and reason == 'entitlement-version-changed'
reason

In [ ]:
scenario = lab.build_scenario()
issued = scenario.grants.issue('session:alice', 'attest:support', lab.root_request('notebook:lifecycle'), now=lab.NOW)
active_agent = scenario.identities.agents['agent:support-7']
scenario.identities.agents['agent:support-7'] = replace(active_agent, lifecycle=lab.Lifecycle.DISABLED)
assert scenario.grants.validate_current(issued.grant, now=lab.NOW)[1] == 'agent-identity-disabled'
scenario.identities.agents['agent:support-7'] = active_agent
affected = scenario.grants.revoke_lineage(issued.grant.grant_id, 'incident-response')
assert affected == 1
assert scenario.grants.validate_current(issued.grant, now=lab.NOW)[1] == 'grant-revoked'
{'revoked_grants': affected, 'terminal_reason': 'grant-revoked'}

In [ ]:
scenario = lab.build_scenario()
scenario.entitlements.available = False
outage = scenario.application.read_attachment(
    session_id='session:alice', agent_attestation_id='attest:support',
    attachment_id='attachment:north:7', request_id='notebook:outage', now=lab.NOW,
)
assert (outage.status, outage.reason) == (lab.DecisionStatus.ERROR, 'entitlement-service-unavailable')
assert outage.content is None and scenario.application.ambient_fallback_count == 0
outage.status.value, outage.reason

## 8. Evaluate safety, utility, replay, and evidence separately

A deny-all control can look safe while destroying utility. The corpus therefore separates valid tasks, attacks and stale-state cases, and dependency failures; it also measures the deliberately unsafe baseline.

In [ ]:
report, observations = lab.evaluate_controls()
assert report.valid_task_success_rate == 1.0
assert report.authority_escalation_success_rate == 0.0
assert report.cross_tenant_disclosure_rate == 0.0
assert report.proof_replay_acceptance_rate == 0.0
assert report.sender_constraint_detection_rate == 1.0
assert report.trace_completeness_rate == 1.0
assert report.baseline_attack_acceptance_rate == 1.0
report

## 9. Put the real OpenAI Agents SDK behind the boundary

The repository-pinned SDK defines a strict `function_tool`, but its model-facing schema contains only `attachment_id`. Human session, workload attestation, managed-agent identity, tenant, purpose, audience, grants, sender proof, policy versions, and correlation IDs remain in trusted application context. The demo invokes the real tool adapter without a model, API key, or network call.

In [ ]:
import sdk_adapter
sdk_proof = await sdk_adapter.credential_free_demo_async()
assert sdk_proof['strict_schema']
assert sdk_proof['direct_terminal_state'] == 'allow'
assert sdk_proof['denied_terminal_state'] == 'deny'
assert sdk_proof['denied_content_absent']
assert sdk_proof['real_tool_invocation_state'] == 'allow'
assert sdk_proof['identity_fields_absent_from_schema']
assert sdk_proof['credential_markers_absent']
sdk_proof

## Production challenge

Replace the teaching HMAC formats with your platform's token exchange, workload identity, key-bound proof, and policy engine. Preserve the executable invariants: immutable subject and tenant; explicit actor chain; exact audience; current lifecycle; monotonic scope, resource, TTL, and depth; one-use request binding; lineage revocation; and zero ambient-authority fallback. Add concurrent replay tests and measure revocation propagation time.

## Checkpoint

An attacker steals a still-unexpired attachment-store grant but cannot authenticate as the named case-service workload or sign with its bound key. Should the store release the attachment?

**Answer:** No. Expiry is only one condition. The store must authenticate the current delegate, match tenant and sender key to the grant, validate a fresh proof bound to that grant and exact request, and recheck current agent, entitlement, policy, lineage, operation, and resource state. The stolen serialized grant is not a bearer credential.